# Exploration of CLIP flaws

Multimodal model can have unexpected biases. CLIP, for example, is strongly biased toward text in the text. A great example can be found [here](https://arxiv.org/pdf/2508.05430#page=8.94), Figure 6, where CLIP "sees" a doll, but is actually focused on "dollar" text, not an actual doll. Is thid a **real** problem? Considering that most of the content on web is watermarked in some way, this might be an issue. During today's lab, we will try to reproduce this phenomena on ImageNet creating artifically injected watermarks.

The coding agenda is as follow:

1. load a CLIP and stream ImageNet dataset
2. create a custom version of dataset with injected watermarks
3. iterate over dataset, compute metrics on original and injected data
4. compute simple CAV (diff means)
5. debias representation of injected data and compare metric to the original data

In [2]:
import os
import numpy as np
from PIL import ImageFont, ImageDraw, Image as PILImage

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 00 Prelimaries

code below serves as utilty to inject watermarks. You need to pass `image` and `text` to be injected, the rest of parameters can be left as default. The watermark transformation can be used via

```python
torchvision.transforms.Compose([partial(add_watermark, ...), clip_text_preprocessor])
```

In [4]:
_IMAGE_SIZE_TO_FONT_SIZE = {
    224: 36,
    384: 62,
    512: 82,
    518: 84,
}
_FONT_SIZE_RATIO = 6.22


def add_watermark(
    image: np.ndarray | PILImage.Image,
    image_size: int = 224,
    text: str = "TEXT",
    font_path: str = "/content/drive/MyDrive/WB2/fonts/SourceHanSerifSC-ExtraLight.otf",
    opacity: float = 0.5,
    color: tuple[int, int, int] = (255, 255, 255),
    x_pos: float = 0.01,
    y_pos: float = 0.4,
) -> PILImage.Image:
    """
    Add semi-transparent text watermark overlay to an image.

    The watermark is composited using alpha blending, preserving the original
    image with the text rendered on top at the specified opacity.

    Args:
        image: Input PIL Image or numpy array to add watermark to
        image_size: Size of input image in pixels, used for font size calculation (default: 224)
        text: Watermark text to overlay (default: WATERMARK_TEXT)
        font_path: Path to TrueType/OpenType font file (default: SourceHanSerifSC-ExtraLight.otf)
        opacity: Watermark opacity in range [0.0, 1.0] where 0.0 is fully transparent
            and 1.0 is fully opaque (default: 0.5)
        color: RGB color tuple for watermark text (default: (255, 255, 255) white)
        x_pos: Horizontal position as fraction of image width in range [0.0, 1.0] (default: 0.01)
        y_pos: Vertical position as fraction of image height in range [0.0, 1.0] (default: 0.4)

    Returns:
        PIL Image with watermark overlay applied, converted to RGB mode

    Raises:
        FileNotFoundError: If font_path doesn't exist
        ValueError: If opacity, x_pos, or y_pos are outside the valid range [0.0, 1.0]
    """
    # Validate font path
    if not os.path.exists(font_path):
        raise FileNotFoundError(f"Font file not found: {font_path}")

    # Validate range parameters
    if not 0.0 <= opacity <= 1.0:
        raise ValueError(f"Opacity must be in range [0.0, 1.0], got {opacity}")
    if not 0.0 <= x_pos <= 1.0:
        raise ValueError(f"x_pos must be in range [0.0, 1.0], got {x_pos}")
    if not 0.0 <= y_pos <= 1.0:
        raise ValueError(f"y_pos must be in range [0.0, 1.0], got {y_pos}")

    # Calculate appropriate font size
    if image_size in _IMAGE_SIZE_TO_FONT_SIZE:
        font_size = _IMAGE_SIZE_TO_FONT_SIZE[image_size]
    else:
        font_size = int(image_size / _FONT_SIZE_RATIO)

    font = ImageFont.truetype(font_path, font_size)

    # If np.array convert to PIL
    if isinstance(image, np.ndarray):
        image = PILImage.fromarray(image.astype(np.uint8), 'RGB')

    # Convert to RGBA for alpha compositing
    image_rgba = image.convert("RGBA")
    width, height = image_rgba.size

    # Create transparent overlay for watermark text
    watermark_layer = PILImage.new("RGBA", image_rgba.size, (255, 255, 255, 0))
    draw = ImageDraw.Draw(watermark_layer)

    # Draw text with specified opacity
    draw.text(
        xy=(x_pos * width, y_pos * height),
        text=text,
        fill=(*color, round(opacity * 255)),  # Cleaner tuple unpacking
        font=font,
    )

    # Composite watermark onto image and convert back to RGB
    output_img = PILImage.alpha_composite(image_rgba, watermark_layer).convert("RGB")
    return output_img

### 01 CLIP and Imagenet loading

You should already know how to load CLIP from previous labs. For ImageNet, you need to find it on HuggingFace. You can check for `imagenet-1k` for smaller volume. Note that "mini" version is still too bug to make the full experiment, so feel free to constraint experiments to 100-200 images. Use `streaming=True`, so the datasrt is not donwloaded on your PC/notebook, but is streamed from web. It is much slower, but fits our needs best.

Side note: `test` split of ImageNet does not have labels.

In [5]:
import torch
from transformers import CLIPProcessor, CLIPModel
from torch.utils.data import IterableDataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
# from huggingface_hub import login
# login(login_key)

In [7]:
from datasets import load_dataset
dataset = load_dataset(
    'imagenet-1k',
    split='validation',
    streaming=True,
    token=True,
    trust_remote_code=True
)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'imagenet-1k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'imagenet-1k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

### 02 Custom dataset

Because of `streaming=True`, you already have `IterableDatset` object. It already has `__getitem__` implemented, so we need to add our custom logic in `__iter__`. The logic should be as follows:

```python
    def __iter__(self):
        for example in self.dataset:
            <your code here>
            yield img, img_corrupted, label
```

Remember that `yield` should return `Tensor`, `dict[str, Tensor]`, or so. Ensure that your custom dataset returns (original image after CLIP processing, image after watermark injection and CLIP processing, original label).

Tip: datasets typically returns index of a class, not its label (e.g., `1`, not `goldfish`). You can still find label using `dataset.features["label"].int2str(IDX)`.

Tip 2: because of streaming, the biggest bottleneck is HuggingFace API, consider setting `batch_size` on `64` or more.

In [8]:
class WatermarkedImageNet(IterableDataset):
    def __init__(self, hf_dataset, processor, max_samples=200, watermark_text="BLUEBERRY"):
        self.dataset = hf_dataset
        self.processor = processor
        self.max_samples = max_samples
        self.watermark_text = watermark_text
        self.features = hf_dataset.features["label"]

    def __iter__(self):
        count = 0
        for item in self.dataset:
            if count >= self.max_samples:
                break

            img = item['image'].convert('RGB')
            label_idx = item['label']
            label_str = self.features.int2str(label_idx)

            img_watermarked = add_watermark(img, image_size=img.size[0], text=self.watermark_text)

            orig_processed = self.processor(images=img, return_tensors="pt")['pixel_values'].squeeze(0)
            wm_processed = self.processor(images=img_watermarked, return_tensors="pt")['pixel_values'].squeeze(0)

            count += 1
            yield orig_processed, wm_processed, label_str

watermark_text = "BLUEBERRY"
custom_dataset = WatermarkedImageNet(dataset, processor, max_samples=100, watermark_text=watermark_text)
dataloader = DataLoader(custom_dataset, batch_size=32)

### 03 Metrics computation

So far, you should have loaded CLIP and iterable dataset. Now it's time to creating **interesting** experiment. The design is up to you, but let's check my proposal below:

> - set watermark text to something like "BLUEBERRY"
> - use CLIP in zero-shot mode (comparing embedding of text and image)
> - check how many images will be predicted as "BLUEBERRY" rather than the original class
>   - e.g., compare cossim(image_watermark, "a photo of a BLUEBERRY") and cossim(image_watermark, "a photo of [ORIGINAL CLASS]")

After this step, you should tell the accuracy of the CLIP in distinguishing between BLUEBERRY and [ORIGINAL CLASS] on original data and data with watermarks (or analogous metrics in our own experiment design).

Tip: use `padding=True` when processing text in batches.

Note: I proposed BLUEBERRY for no particural reason, but keep in mind tht ImageNet contains ~all types of images so to be completly fair in this experiment we should choose something that does not occur in this dataset probably.

In [ ]:
orig_embeddings = []
wm_embeddings = []
labels = []

model.eval()
with torch.no_grad():
    for orig_imgs, wm_imgs, batch_labels in tqdm(dataloader):
        orig_imgs = orig_imgs.to(device)
        wm_imgs = wm_imgs.to(device)

        # Pobieranie embeddingów dla oryginałów
        orig_out = model.get_image_features(pixel_values=orig_imgs)
        if not isinstance(orig_out, torch.Tensor):
            orig_emb = getattr(orig_out, "image_embeds", getattr(orig_out, "pooler_output", orig_out[0]))
        else:
            orig_emb = orig_out

        orig_emb = orig_emb / orig_emb.norm(p=2, dim=-1, keepdim=True)

        # Pobieranie embeddingów dla obrazów ze znakiem wodnym
        wm_out = model.get_image_features(pixel_values=wm_imgs)
        if not isinstance(wm_out, torch.Tensor):
            wm_emb = getattr(wm_out, "image_embeds", getattr(wm_out, "pooler_output", wm_out[0]))
        else:
            wm_emb = wm_out

        wm_emb = wm_emb / wm_emb.norm(p=2, dim=-1, keepdim=True)

        orig_embeddings.append(orig_emb.cpu())
        wm_embeddings.append(wm_emb.cpu())
        labels.extend(batch_labels)

orig_embeddings = torch.cat(orig_embeddings)
wm_embeddings = torch.cat(wm_embeddings)

# Embedding dla wstrzykniętego tekstu
text_blueberry = f"a photo of a {watermark_text}"
text_blueberry_inputs = processor(text=[text_blueberry], return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    emb_blueberry_out = model.get_text_features(**text_blueberry_inputs)
    if not isinstance(emb_blueberry_out, torch.Tensor):
        emb_blueberry = getattr(emb_blueberry_out, "text_embeds", getattr(emb_blueberry_out, "pooler_output", emb_blueberry_out[0]))
    else:
        emb_blueberry = emb_blueberry_out

    emb_blueberry = (emb_blueberry / emb_blueberry.norm(p=2, dim=-1, keepdim=True)).cpu()

text_classes = [f"a photo of a {label}" for label in labels]
text_classes_inputs = processor(text=text_classes, return_tensors="pt", padding=True, truncation=True).to(device)
with torch.no_grad():
    emb_classes_out = model.get_text_features(**text_classes_inputs)
    if not isinstance(emb_classes_out, torch.Tensor):
        emb_classes = getattr(emb_classes_out, "text_embeds", getattr(emb_classes_out, "pooler_output", emb_classes_out[0]))
    else:
        emb_classes = emb_classes_out

    emb_classes = (emb_classes / emb_classes.norm(p=2, dim=-1, keepdim=True)).cpu()

orig_correct = 0
wm_correct = 0

for i in range(len(labels)):
    # Dla oryginalnego obrazka:
    sim_orig_class_orig_img = orig_embeddings[i] @ emb_classes[i].T
    sim_blueberry_orig_img = orig_embeddings[i] @ emb_blueberry.T
    if sim_orig_class_orig_img > sim_blueberry_orig_img:
        orig_correct += 1

    # Dla zepsutego obrazka (ze znakiem wodnym):
    sim_orig_class_wm_img = wm_embeddings[i] @ emb_classes[i].T
    sim_blueberry_wm_img = wm_embeddings[i] @ emb_blueberry.T
    if sim_orig_class_wm_img > sim_blueberry_wm_img:
        wm_correct += 1

print(f"Oryginalne obrazy rozpozane poprawnie (vs BLUEBERRY): {orig_correct / len(labels):.2%}")
print(f"Obrazy ze znakiem wodnym rozpoznane poprawnie (vs BLUEBERRY): {wm_correct / len(labels):.2%}")

Ekstrakcja cech (embeddings) z modeli wizyjnych...


4it [00:36,  9.07s/it]


Oryginalne obrazy rozpozane poprawnie (vs BLUEBERRY): 100.00%
Obrazy ze znakiem wodnym rozpoznane poprawnie (vs BLUEBERRY): 93.00%


/tmp/ipykernel_5447/320986027.py:74: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4480.)
  sim_orig_class_orig_img = orig_embeddings[i] @ emb_classes[i].T


### 04 CAV computation

In this step, you should create a CAV (e.g., DiffMeans we used in previous work) and check its detection power - i.e., `rocauc(cossim(CAV, data))`. This should be really easy!

In [11]:
mean_wm = wm_embeddings.mean(dim=0)
mean_orig = orig_embeddings.mean(dim=0)

cav = mean_wm - mean_orig
cav = cav / cav.norm(p=2, dim=-1, keepdim=True)

all_embs = torch.cat([orig_embeddings, wm_embeddings])
y_true = np.array([0] * len(orig_embeddings) + [1] * len(wm_embeddings))

cossims = (all_embs @ cav.T).squeeze().numpy()

roc_auc = roc_auc_score(y_true, cossims)
print(f"ROC-AUC wykrywania znaku wodnego za pomocą wektora CAV: {roc_auc:.4f}")

ROC-AUC wykrywania znaku wodnego za pomocą wektora CAV: 0.9896


### 05 Debiasing

Now we have everything we need to make CLIP robust to watermarks. In this step, repeat the logic from **03**, but make an ortogonalisation of data with injected watermark. Ideally, its CLIP's embedding should be similar to the original data (you can check!); however, our main goal is to just improve the accuracy of zero-shot classification.

In [12]:
wm_debiased = wm_embeddings - (wm_embeddings @ cav.T).view(-1, 1) * cav

wm_debiased = wm_debiased / wm_debiased.norm(p=2, dim=-1, keepdim=True)

debiased_correct = 0

for i in range(len(labels)):
    sim_orig_class_debiased_img = wm_debiased[i] @ emb_classes[i].T
    sim_blueberry_debiased_img = wm_debiased[i] @ emb_blueberry.T

    if sim_orig_class_debiased_img > sim_blueberry_debiased_img:
        debiased_correct += 1

print(f"Obrazy PO DEBIASINGU rozpoznane poprawnie (vs BLUEBERRY): {debiased_correct / len(labels):.2%}")

Obrazy PO DEBIASINGU rozpoznane poprawnie (vs BLUEBERRY): 98.00%


93 -> 98 mamy poprawę :)